In [0]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from utils.utils_merge_into_tables import upsert_data
from utils.utils_transform_data import cast_columns, standardize_column_names, standardize_string_values
from pyspark.sql import functions as sf 



In [0]:
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
table_bronze = dbutils.widgets.get("table_bronze")
table_silver = dbutils.widgets.get("table_silver")
primary_keys = dbutils.widgets.get("primary_keys")

In [0]:

df_bronze = spark.read.table(f"{catalog}.{schema_bronze}.{table_bronze}")


## Transform Data


In [0]:
df_bronze = df_bronze.select("order_id","order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value")

data_type_mapping = {
    "order_id": "string",
    "order_item_id": "integer",
    "product_id": "string",
    "seller_id": "string",
    "shipping_limit_date": "timestamp",
    "price": "double",
    "freight_value": "double",
}

In [0]:
df_silver = (
    df_bronze
    .transform(lambda df: cast_columns(df, data_type_mapping))
    .transform(standardize_column_names)
    .transform(standardize_string_values)
    .withColumn("silver_update_date", sf.current_timestamp())
)

## Merge Table

In [0]:
%run ../setup/00_aws_connection

In [0]:
full_table = f"{catalog}.{schema_silver}.{table_silver}"

upsert_data(df_silver,full_table,primary_keys,name_bucket,layer="silver")